# MuRIL and XLM-R: Hindi-English emotion study

Fine-tune MuRIL in **Colab with a GPU**, using the same 1,000 Hindi-English test phrases as the existing combined XLM-R run. Select **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. Upload `muril_study.zip` when prompted. This is supervised seven-emotion fine-tuning, not pretraining a language model from scratch.

The data is synthetic and template-based. These scores do not demonstrate real-world accuracy. The existing combined XLM-R had 16,000 training phrases across Hindi-English and English-French; MuRIL uses 8,000 Hindi-English phrases. An optional matched Hindi-English XLM-R run helps separate the encoder difference from training-language scope. Run MuRIL first, then the optional matched job in a separate session.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import files, drive
uploaded=files.upload()
zips=[n for n in uploaded if n.lower().endswith(".zip")]
assert len(zips)==1, "Upload exactly one generated training ZIP"
base=Path("/content").resolve()
with zipfile.ZipFile(zips[0]) as z:
    assert sum(i.file_size for i in z.infolist()) < 512*1024**2
    for i in z.infolist():
        assert "\\" not in i.filename and (base/i.filename).resolve().is_relative_to(base)
        assert (i.external_attr>>16)&0o170000 != 0o120000
    z.extractall(base)
PACKAGE=base/"hinglish_project_training"
m=json.loads((PACKAGE/"MANIFEST.json").read_text())
for name,digest in m["sha256"].items():
    p=(PACKAGE/name).resolve(); assert p.is_relative_to(PACKAGE)
    assert hashlib.sha256(p.read_bytes()).hexdigest()==digest, name
PROJECT=PACKAGE/"source"; os.chdir(PROJECT)
subprocess.run([sys.executable,"-m","pip","install","-e",".[model]"],check=True)
import torch
assert torch.cuda.is_available(), "Enable Runtime > Change runtime type > T4 GPU, then restart from this cell"
print("GPU:",torch.cuda.get_device_name(0))


In [ ]:
# One job per session. For the optional matched run, change JOB only.
JOB = "muril_hi_en"  # muril_hi_en | xlmr_hi_en_matched
EPOCHS = 3
assert JOB in {"muril_hi_en", "xlmr_hi_en_matched"}
assert EPOCHS == 3, "Keep the study's three-epoch maximum (early stopping uses validation only)."
drive.mount("/content/drive")
STORAGE = Path("/content/drive/MyDrive/HinglishEmotionProject/MuRILStudy")
run = STORAGE / JOB
run.mkdir(parents=True, exist_ok=True)
model = "muril" if JOB == "muril_hi_en" else "xlmr"
DATA = PACKAGE / "data/all_splits.csv"
AUDIT = PACKAGE / "data/dataset_audit.json"
contract = json.loads((PACKAGE / "study/study_contract.json").read_text())
baseline = json.loads((PACKAGE / "study/xlmr_baseline_metrics.json").read_text())


## Freeze and check the data

Only Hindi-English rows enter either new job. The stored hashes bind the text, labels, row identities and splits to the installed XLM-R baseline. Training updates weights on 8,000 rows; 1,000 validation rows select the best checkpoint; 1,000 held-out test rows are evaluated after selection. Do not tune settings using the test scores. The labels are joy, sadness, anger, fear, surprise, disgust and neutral.


In [ ]:
from training.data_workflow import prepare_data
from training.muril_study import test_fingerprint, validate_baseline
validate_baseline(contract, baseline)
prepared = prepare_data(DATA, "emotion", "raw", 42, run / "frozen_split.csv", AUDIT, "hi-en", allow_synthetic=True)
expected = contract["matched"]
assert prepared["dataset_sha256"] == expected["dataset_sha256"], "Dataset changed"
assert prepared["manifest_sha256"] == expected["split_sha256"], "Split changed"
assert test_fingerprint(prepared["parts"][2]) == contract["hi_en_test_sha256"], "Test phrases changed"
sizes = dict(zip(("train", "validation", "test"), (len(part) for part in prepared["parts"])))
assert sizes == expected["split_sizes"]
print("Job:", JOB, "| Synthetic Hindi-English splits:", sizes)


## Train on the GPU

Both new runs use raw text, length 128, batch size 2, eight accumulation steps, learning rate 0.00002, seed 42, balanced class weights and fp16. All encoder weights and the emotion head are fine-tuned. Validation macro-F1 selects the best checkpoint, with patience 2 and at most three epochs. Epoch recovery state and the frozen split save to Drive. After a runtime reset, use the same ZIP and JOB, then Run all; completed epochs can be resumed. CPU training is intentionally disabled.

Model cards: [MuRIL](https://huggingface.co/google/muril-base-cased), [XLM-R](https://huggingface.co/FacebookAI/xlm-roberta-base). MuRIL is restricted to Hindi-English in this project.


In [ ]:
command = [sys.executable, "-u", "-m", "training.compare_encoders",
    "--task", "emotion", "--models", model, "--data", str(DATA), "--language-pair", "hi-en",
    "--output-root", str(run / "models"), "--split-manifest", str(run / "frozen_split.csv"),
    "--epochs", str(EPOCHS), "--batch-size", "2", "--gradient-accumulation", "8",
    "--max-length", "128", "--learning-rate", "2e-5", "--precision", "fp16",
    "--text-mode", "raw", "--seed", "42", "--split-seed", "42", "--resume",
    "--data-audit", str(AUDIT), "--allow-synthetic"]
subprocess.run(command, cwd=PROJECT, check=True)


## Download the trained checkpoint

The export contains the best inference weights, tokenizer, configuration and metrics. Training data and optimizer recovery files remain outside the export. Keep the full Drive job folder if you want to resume. On your Windows computer, install `muril_hi_en.zip` with `python -m training.install_checkpoint "C:/Users/Advay/Downloads/muril_hi_en.zip"` from the project folder. The dashboard will discover it under MuRIL. The optional matched XLM-R export installs separately from the existing combined model.


In [ ]:
from training.cloud_bundle import export_checkpoint
model_dir = run / "models" / (model + "_emotion")
export_path = STORAGE / "exports" / (JOB + ".zip")
export_checkpoint(model_dir, export_path)
print("Inference checkpoint:", export_path)
files.download(str(export_path))


## Compare the saved runs

Accuracy counts correct test predictions. Macro-F1 gives each emotion equal weight. The report includes both metrics, per-emotion precision/recall/F1/support, a graph and the study limitations. It uses the baseline's Hindi-English score, not its combined-language overall score. Mismatched data, splits, labels or declared settings are rejected. This is a single-seed synthetic study; no statistical-significance or real-world-performance claim follows from it.


In [ ]:
from training.muril_study import compare_runs, write_report
from IPython.display import SVG, display
import pandas as pd
muril_path = STORAGE / "muril_hi_en/models/muril_emotion/metrics.json"
matched_path = STORAGE / "xlmr_hi_en_matched/models/xlmr_emotion/metrics.json"
if not muril_path.is_file():
    print("Train the muril_hi_en job first. No MuRIL score is available yet.")
else:
    muril = json.loads(muril_path.read_text())
    matched = json.loads(matched_path.read_text()) if matched_path.is_file() else None
    report = compare_runs(contract, baseline, muril, matched)
    report_dir = write_report(report, run / "comparison")
    display(pd.DataFrame(report["rows"]))
    display(SVG(filename=str(report_dir / "comparison.svg")))
    display(pd.DataFrame(report["per_class"]))
    for note in report["notes"]:
        print(note)
    report_zip = STORAGE / "exports" / (JOB + "_comparison.zip")
    with zipfile.ZipFile(report_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for file in sorted(report_dir.iterdir()):
            if file.is_file():
                z.write(file, file.name)
    files.download(str(report_zip))
